# 🎬 Wan 2.2 Image-to-Video — ComfyUI on Colab (A100)

Run the **Wan 2.2 14B I2V** model in **ComfyUI** on a Colab **A100**, served through a public URL (Colab proxy by default, with Cloudflare/ngrok options). Models live in Google Drive so they download **once** and are reused on every session.

Wan 2.2 14B uses a dual-expert (MoE) design: a **high-noise** model handles early denoising steps and a **low-noise** model finishes — better motion coherence than Wan 2.1, at the cost of loading two diffusion models.

| Component | File | ComfyUI folder |
|---|---|---|
| High-noise expert | `wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors` (~13 GB) | `diffusion_models` |
| Low-noise expert | `wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors` (~13 GB) | `diffusion_models` |
| Text encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` (~6 GB) | `text_encoders` |
| VAE | `wan_2.1_vae.safetensors` (~242 MB) | `vae` |
| CLIP vision* | `clip_vision_h.safetensors` (~1.2 GB) | `clip_vision` |

> ⚠️ Use **`wan_2.1_vae.safetensors`** — *not* the 2.2 VAE — for the 14B I2V workflow.
> *CLIP vision is only needed for Wan 2.1-style I2V graphs; the native Wan 2.2 14B I2V template does not require it, but it's kept for compatibility.

**Setup:** `Runtime → Change runtime type → A100 GPU`. Colab A100 = 40 GB VRAM; fp8 I2V uses ~30 GB.

## Step 1 — Verify GPU

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
if 'A100' in gpu:
    print('✅ A100 confirmed — up to 161 frames (~10s) is safe')
elif 'L4' in gpu:
    print('⚠️  L4 — keep frames ≤ 113 (~7s) and use 480p')
elif 'T4' in gpu:
    print('⚠️  T4 (16 GB) is too small for 14B fp8. Switch to A100: Runtime → Change runtime type → A100 GPU')
else:
    print('⚠️  Recommended: A100. Runtime → Change runtime type → A100 GPU')

## Step 2 — Mount Google Drive

Creates the persistent model/output folders under `MyDrive/ComfyUI_Wan` (matches your existing layout, so already-downloaded models are reused).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ComfyUI_Wan'
MODELS_DIR = f'{DRIVE_BASE}/models'

for d in [f'{MODELS_DIR}/diffusion_models', f'{MODELS_DIR}/text_encoders',
          f'{MODELS_DIR}/vae', f'{MODELS_DIR}/clip_vision', f'{MODELS_DIR}/loras',
          f'{DRIVE_BASE}/output', f'{DRIVE_BASE}/input_images']:
    os.makedirs(d, exist_ok=True)

print(f'✅ Drive mounted: {DRIVE_BASE}')

## Step 3 — Install ComfyUI + custom nodes

Native Wan 2.2 support is built into recent ComfyUI core. Adds ComfyUI-Manager and VideoHelperSuite (for video preview/save nodes).

In [ ]:
import os, subprocess
os.chdir('/content')

# ComfyUI — clone fresh if missing/corrupt, else update
if os.path.exists('/content/ComfyUI'):
    ok = subprocess.run(['git', 'rev-parse', '--git-dir'], cwd='/content/ComfyUI',
                        capture_output=True).returncode == 0
    if ok:
        !cd /content/ComfyUI && git pull -q
        print('✅ ComfyUI updated')
    else:
        !rm -rf /content/ComfyUI && git clone -q https://github.com/comfyanonymous/ComfyUI.git
        print('✅ ComfyUI re-cloned (was corrupt)')
else:
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
    print('✅ ComfyUI cloned')

# Core requirements (flag avoids reinstalling every session)
if not os.path.exists('/content/comfyui_reqs_installed'):
    !pip install -q -r /content/ComfyUI/requirements.txt
    open('/content/comfyui_reqs_installed', 'w').close()
    print('✅ Requirements installed')
else:
    print('✅ Requirements already installed')

# Custom nodes
for name, repo in [('ComfyUI-Manager', 'https://github.com/ltdrdata/ComfyUI-Manager.git'),
                   ('ComfyUI-VideoHelperSuite', 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')]:
    path = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(path):
        !git clone -q {repo} {path}
        if os.path.exists(f'{path}/requirements.txt'):
            !pip install -q -r {path}/requirements.txt
        print(f'✅ {name} installed')
    else:
        !cd {path} && git pull -q
        print(f'✅ {name} ready')

# ffmpeg (for video utilities)
if subprocess.run(['which', 'ffmpeg'], capture_output=True).returncode != 0:
    !apt-get install -y -q ffmpeg
print('\n✅ All installs complete')

## Step 4 — Link Drive model folders into ComfyUI

Symlinks each `models/<folder>` and `output` to Drive, so ComfyUI reads the already-downloaded weights and writes renders straight to Drive.

In [ ]:
import os
COMFY_MODELS = '/content/ComfyUI/models'

links = {f'{COMFY_MODELS}/{f}': f'{MODELS_DIR}/{f}'
         for f in ['diffusion_models', 'text_encoders', 'vae', 'clip_vision', 'loras']}
links['/content/ComfyUI/output'] = f'{DRIVE_BASE}/output'

for dst, src in links.items():
    if os.path.exists(dst) and not os.path.islink(dst):
        !rm -rf {dst}
    if not os.path.islink(dst):
        os.symlink(src, dst)
    print(f'  ✅ {os.path.basename(dst)} → Drive')
print('\n✅ Folders linked')

## Step 5 — Download Wan 2.2 models (skips anything already in Drive)

Your Drive already has these as `fp8_scaled`, so this should report **all present** and download nothing. First-ever run pulls ~34 GB (~30 min).

In [ ]:
import os
HF22 = 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files'
HF21 = 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'

models = [
    ('Wan 2.2 I2V High Noise fp8 (~13GB)',
     f'{HF22}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors',
     f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors'),
    ('Wan 2.2 I2V Low Noise fp8 (~13GB)',
     f'{HF22}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors',
     f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors'),
    ('Text Encoder UMT5 fp8 (~6GB)',
     f'{HF21}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
     f'{MODELS_DIR}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
    ('VAE wan_2.1_vae (~242MB)',
     f'{HF21}/vae/wan_2.1_vae.safetensors',
     f'{MODELS_DIR}/vae/wan_2.1_vae.safetensors'),
    ('CLIP Vision (~1.2GB)',
     f'{HF21}/clip_vision/clip_vision_h.safetensors',
     f'{MODELS_DIR}/clip_vision/clip_vision_h.safetensors'),
]

for label, url, dest in models:
    if os.path.exists(dest) and os.path.getsize(dest) > 1024:
        print(f'  ✅ Present ({os.path.getsize(dest)/1024**3:.1f}GB): {label}')
    else:
        print(f'  ⬇️  Downloading: {label}')
        !wget -q --show-progress -O "{dest}" "{url}"
        print(f'  ✅ Done ({os.path.getsize(dest)/1024**3:.1f}GB): {label}')
print('\n✅ All models ready')

## Step 6 — *(Optional)* Lightning 4-step LoRAs — ~6× faster

[lightx2v Wan2.2-Lightning](https://huggingface.co/lightx2v/Wan2.2-Lightning) LoRAs let you render in **4–8 steps** at **CFG 1.0** instead of 20 steps. Separate LoRA per expert. This auto-discovers the current I2V filenames and saves them to Drive. Skip for maximum quality.

In the workflow: add a `LoraLoaderModelOnly` on **each** expert (HIGH lora → high-noise model, LOW lora → low-noise model), set steps **4–8**, CFG **1.0**, sampler **euler**, scheduler **simple**.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

LORA_REPO = 'lightx2v/Wan2.2-Lightning'
lora_dir = f'{MODELS_DIR}/loras'
try:
    files = HfApi().list_repo_files(LORA_REPO)
    cand = [f for f in files if f.endswith('.safetensors') and 'I2V' in f.upper()
            and '4STEP' in f.upper().replace('-', '').replace('_', '')]
    if not cand:
        cand = [f for f in files if f.endswith('.safetensors') and 'I2V' in f.upper()]
    picks = []
    for tag in ('HIGH', 'LOW'):
        m = [f for f in cand if tag in f.upper()]
        if m:
            picks.append(sorted(m)[0])
    picks = picks or cand[:2]
    print('Selected:', *picks, sep='\n  ')
    for src in picks:
        dest = f'{lora_dir}/{os.path.basename(src)}'
        if os.path.exists(dest):
            print(f'  ✅ Present: {os.path.basename(src)}')
            continue
        cached = hf_hub_download(repo_id=LORA_REPO, filename=src)
        import shutil; shutil.copy(cached, dest)
        print(f'  ✅ Downloaded: {os.path.basename(src)}')
    print('\n✅ Lightning LoRAs in Drive models/loras/')
except Exception as e:
    print('Could not auto-download Lightning LoRAs:', e)
    print('Install them via ComfyUI-Manager instead, or skip this step.')

## Step 7 — Launch ComfyUI + public URL

Starts ComfyUI, waits until it's actually serving, then exposes it. Pick a `TUNNEL` method:

- **`colab`** *(default — no auth)*: opens ComfyUI as a **clickable "new window" link + an embedded iframe** inside this cell's output. ⚠️ Do **not** copy the raw `...prod.colab.dev` URL into a separate browser — it only works inside this Colab session and otherwise returns **HTTP 404**. Use the link/iframe this cell renders.
- **`cloudflare`**: quick `trycloudflare.com` tunnel that works in any browser. The cell first deletes any stale `~/.cloudflared` credentials (the usual cause of *"authentication"* errors) and installs a fresh binary.
- **`ngrok`**: paste a free authtoken from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

If ComfyUI itself fails to start, the tail of its log is shown. **Keep this cell running.**

In [ ]:
#@title Step 7 — Launch ComfyUI + tunnel { display-mode: "form" }
TUNNEL = "colab"  #@param ["colab", "cloudflare", "ngrok"]
NGROK_TOKEN = ""  #@param {type:"string"}

import os, re, time, subprocess, urllib.request

PORT = 8188
LOG = '/tmp/comfyui.log'

# --- start ComfyUI (kill any previous run first) ---
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--preview-method', 'auto'],
    cwd='/content/ComfyUI', stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

print('Waiting for ComfyUI to start...')
ready = False
for _ in range(60):
    time.sleep(2)
    if comfy.poll() is not None:
        print('\n❌ ComfyUI exited. Last log lines:\n')
        print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
        raise SystemExit('ComfyUI failed to start — see log above.')
    try:
        if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2).status == 200:
            ready = True
            break
    except Exception:
        pass
if not ready:
    print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
    raise SystemExit('ComfyUI did not become ready in time — see log above.')
print('✅ ComfyUI is serving on :%d' % PORT)

def banner(url, note=''):
    print('\n' + '=' * 64)
    print('  🚀  Open ComfyUI:  ' + url)
    if note:
        print('  ' + note)
    print('=' * 64)

tunnel = None

if TUNNEL == 'colab':
    # IMPORTANT: the raw https://<...>.prod.colab.dev URL only works inside THIS
    # Colab tab's session (it needs the service worker this page registers).
    # Pasting it into a different browser/tab gives HTTP 404. So we open it via
    # Colab's own helpers: a "new window" link + an embedded iframe — both keep
    # the right session context.
    from google.colab import output
    print('▶ Click this link to open ComfyUI in a new tab:')
    output.serve_kernel_port_as_window(PORT)
    print('\n▶ ...or use ComfyUI embedded right here:')
    output.serve_kernel_port_as_iframe(PORT, height='820')
    print('\n(If the embed/link misbehaves, set TUNNEL="cloudflare" and re-run for a normal public URL.)')

elif TUNNEL == 'ngrok':
    if not NGROK_TOKEN:
        raise SystemExit('Set NGROK_TOKEN in the form, or use TUNNEL="colab".')
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    banner(ngrok.connect(PORT, 'http').public_url, '(ngrok)')

elif TUNNEL == 'cloudflare':
    # Clear stale named-tunnel creds (the usual "authentication" cause), fresh binary.
    !rm -rf ~/.cloudflared
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_re = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    found = False
    t0 = time.time()
    for line in tunnel.stdout:
        m = url_re.search(line)
        if m:
            banner(m.group(0), '(403? open the link in an incognito window)')
            found = True
            break
        if time.time() - t0 > 40:
            break
    if not found:
        print('⚠️  Cloudflare returned no URL (auth/network). Set TUNNEL="colab" and re-run.')

print('\n⏳ Keep this cell running. Interrupt to stop.')
try:
    comfy.wait()
except KeyboardInterrupt:
    comfy.terminate()
    if tunnel:
        tunnel.terminate()
    print('\n🛑 ComfyUI stopped')

## Step 8 — Run the I2V workflow

In the ComfyUI tab:
1. **Load template:** `Workflow → Browse Templates → Video → Wan2.2 14B I2V`.
2. Confirm the loaders point at your files: high/low noise `UNETLoader`s, `CLIPLoader` = `umt5_xxl_fp8_e4m3fn_scaled`, `VAELoader` = **`wan_2.1_vae.safetensors`**.
3. **Load Image** node = your first frame (use the *Copy Input Image* utility below, or upload via the UI).
4. Write the motion/scene in the positive prompt.
5. **Settings (A100):** 832×480 (fast) or 1280×720 (HQ); 81 frames @ 16 fps (~5 s); 20 steps, CFG 3.5. *With Lightning LoRAs:* 4–8 steps, CFG 1.0.
6. **Run.** Output MP4 lands in Drive `ComfyUI_Wan/output/`.

**OOM on 40 GB?** Use 480p, fewer frames, and add `--lowvram` to the launch command in Step 7.

---
## 🔧 Utilities

### Copy an input image into ComfyUI

In [ ]:
import shutil, os
SOURCE = f'{DRIVE_BASE}/input_images/my_image.jpg'  # update filename
if os.path.exists(SOURCE):
    shutil.copy(SOURCE, f'/content/ComfyUI/input/{os.path.basename(SOURCE)}')
    print(f'✅ Copied: {os.path.basename(SOURCE)} (pick it in the Load Image node)')
else:
    print(f'⚠️  Not found: {SOURCE}\n   Upload to Drive → ComfyUI_Wan → input_images')

### Extract last frame (to chain clips)

In [ ]:
import shutil
CLIP_IN    = f'{DRIVE_BASE}/output/wan22_i2v_00001.mp4'  # update filename
LAST_FRAME = f'{DRIVE_BASE}/input_images/last_frame.jpg'
!ffmpeg -y -sseof -1 -i "{CLIP_IN}" -frames:v 1 "{LAST_FRAME}"
shutil.copy(LAST_FRAME, '/content/ComfyUI/input/last_frame.jpg')
print('✅ Last frame ready in ComfyUI input')

### Concatenate clips / add audio

In [ ]:
import glob
clips = sorted(glob.glob(f'{DRIVE_BASE}/output/wan22_i2v_*.mp4'))
print(f'Found {len(clips)} clips')
if len(clips) >= 2:
    OUT = f'{DRIVE_BASE}/output/wan22_i2v_combined.mp4'
    with open('/tmp/concat.txt', 'w') as f:
        for c in clips:
            f.write(f"file '{c}'\n")
    !ffmpeg -y -f concat -safe 0 -i /tmp/concat.txt -c copy "{OUT}"
    print(f'✅ Combined → {OUT}')
else:
    print('⚠️  Need ≥ 2 clips to concatenate')

---
## 📋 Prompt & troubleshooting tips

**Frames:** 81 ≈ 5s · 113 ≈ 7s · 145 ≈ 9s · 161 ≈ 10s (@16fps)

**Prompts** — Wan 2.2 follows direction well; be specific about motion and camera:
- Motion: `"she turns, walks forward, pauses and looks back"`
- Camera: `"slow push in"`, `"gentle pan right"`, `"tilt up to sky"`
- Quality: `"cinematic, smooth motion, film grain, 4k"`
- Negative: `"low quality, blurry, distorted, static, no motion, watermark, jitter"`

**Troubleshooting**
- **Cloudflare "authentication" / 403** → use `TUNNEL = "colab"` in Step 7 (no auth, no external service), or just use the *Backup (Colab proxy)* link the cell prints. The Cloudflare path also wipes stale `~/.cloudflared` creds, which are the usual cause.
- **VAE errors** → Load VAE must be `wan_2.1_vae.safetensors`, not the 2.2 VAE.
- **Missing nodes** → Step 3 does `git pull`; restart ComfyUI so core picks up Wan 2.2 nodes.
- **OOM** → 480p, fewer frames, `--lowvram` in Step 7.
- **Cell stops immediately** → Step 7 prints the ComfyUI log tail on failure; read it for the real error.